# Q1D | Train Linear Regression | Focused Tuning

> Takes the winning preprocessing from D-LinearRegression-Combined_Stage_Experiments.ipynb (experiments F to I) and tunes the models' settings on it with nested cross-validation, in a new suite of experiments J+

# Imports

In [1]:
from pathlib import Path

import sklearn
print(sklearn.__version__)

import joblib
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, clone
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline

from Q1.consts import *
from utils.read_csv_to_df import read_csv_to_df
from Q1.utils.create_pipeline import create_preprocessing, preprocessing_name
from Q1.utils.run_experiment import build_pipelines, cross_validate_folds, model_file_name
from Q1.utils.complete_row_folds import Folds, complete_row_folds

from Q1.models import MODELS
from Q1.performance_measures import SCORERS
from Q1.utils.preprocessing.scale import StandardScaler
from Q1.utils.preprocessing.transform import PowerTransformer, on_columns

1.3.2


# Defaults

In [2]:
SEED = 42
np.random.seed(SEED)
pd.set_option("display.float_format", "{:,.4f}".format)

# changing only the fold seed moves mean RMSE by about this much, so smaller gaps are ties
NOISE_FLOOR = 0.002

# paired fold gaps smaller than this are floating-point rounding, not a real difference, so they count as ties instead of wins
WIN_TOLERANCE = 1e-9

# ConvergenceWarnings are left on: if Lasso or ElasticNet don't converge at a small alpha, that should show

# Shared Setup

> Every experiment in this notebook (J to M) follows the rules below, so their scores compare directly with each other and with F to I

> **Preprocessing (fixed, I's winner):** drop incomplete rows → Yeo-Johnson (`standardize=False`) on the 6 skewed columns → `StandardScaler`, no imputer. The skewed columns use the Q1B rule, |skew| ≥ 0.5 on the full training X. Only the model settings change in this notebook

> **Scoring:** the same as I. Every model is scored with `SCORERS` (RMSE, MSE, MAE) through `cross_validate_folds` (Q1/utils/run_experiment.py), per fold and unrounded, on `complete_row_folds(X_clean, seed=s, add_incomplete_to_train=False)` for the fold seeds {42, 0, 1, 2, 3}: 5 seeds x 4 folds = 20 outer folds. Within a fold every model validates on the same rows, so each fold is a paired comparison. Seed 42 gives the folds of A to I

> **Nested cross-validation:** each tuned model is a `GridSearchCV(pipeline, param_grid, cv=KFold(n_splits=4, shuffle=True, random_state=SEED), scoring=SCORERS["rmse"], refit=True, n_jobs=1)`, and that whole search is the estimator `cross_validate_folds` scores
> - **Inner loop:** on each outer training fold only, the search splits those rows into 4 inner folds, picks the setting with the lowest inner RMSE, and refits it on the whole outer training fold
> - **Outer loop:** that refit model is scored on the outer validation fold, which played no part in picking the setting
> - **Why:** picking a setting by its CV score and then reporting that same score is optimistic, since the pick favors whichever setting got lucky on those folds. The gaps here are about 0.001, so that bias alone could manufacture a win. Nested CV scores the whole tuning procedure on rows it never saw, so a tuned model and the untuned baseline are judged the same way
> - `n_jobs=1` inside, because `cross_validate_folds` already runs the outer folds in parallel

> **Chosen settings:** `cross_validate_folds` keeps the scores but not the fitted models, so for each model and outer fold a copy of the same search is fit on that fold's training rows to read its `best_params_`. Its RMSE on the fold's validation rows must equal the `cross_validate_folds` score for that fold (asserted), which proves both are the same procedure. A chosen value at the min or max of its grid is flagged as **edge**: the best value may lie past it, so that grid should be widened before the setting is trusted

> **From K on:** `score_folds_with_settings` reads each fold's setting from the fitted search that `cross_validate_folds` returns (`return_estimator=True`) instead of refitting it, so the setting comes from the very model that was scored; J keeps the refit and assert above

> **Baseline and bar:** the untuned linear_regression on the fixed preprocessing, scored on the same 20 folds (0.5258 in I). A tuned model counts as better only if it beats the baseline by more than the 0.002 noise floor in mean RMSE over the 20 folds

> **Reported per model:** mean RMSE / MSE / MAE over the 20 folds, "seed std" (the std of the 5 seed means), "gap" vs the baseline (model minus baseline, so negative means better), "wins" (folds where the gap is below -`WIN_TOLERANCE`, out of 20), and the settings chosen across the 20 folds with their edge flags

> **Final models:** each tuned model's search is refit on all 1,374 complete rows, and its `best_estimator_` (the preprocessing + the model at the chosen setting) is saved, with the chosen setting recorded. The final setting is picked on all the rows, so it can differ from the per-fold picks

> **Model-simplicity order for M (decided before any results):** the best mean RMSE over the 20 folds wins; if the top models are within 0.002, take the simplest in this order:
> 1. linear_regression
> 2. ridge
> 3. Lasso / ElasticNet
> 4. SGD
> 5. svr_linear / LinearSVR

> **No test set:** only the training set is used in this notebook. The test set is scored once, in a later step, after M picks the model

## Plan

| experiment | change                                             | question                                                                                                              |
|------------|----------------------------------------------------|-----------------------------------------------------------------------------------------------------------------------|
| J          | Regularization strength (ridge, Lasso, ElasticNet) | does a tuned alpha bring Lasso and ElasticNet up to OLS, and does any regularized model beat OLS by more than 0.002?   |
| K          | SVR margin settings (svr_linear, LinearSVR)        | does tuning the margin settings close svr_linear's and LinearSVR's gap to OLS (0.0025 and 0.0035 in I)?                |
| L          | SGD                                                | does tuning SGD's settings beat OLS by more than 0.002?                                                                |
| M          | Final selection                                    | which model goes to the test set, by best mean RMSE and the model-simplicity order above?                              |

## Shared Helpers

> defined once here so every experiment (J to M) builds its searches, scores, and tables the same way

In [3]:
# the inner CV of every search: 4 seeded folds of the outer training fold it is fit on
INNER_CV = KFold(n_splits=4, shuffle=True, random_state=SEED)

def nested_search(pipeline: Pipeline, param_grid: dict) -> GridSearchCV:
    # picks the setting with the lowest inner RMSE on the rows it is fit on, then refits it on all of them.
    # n_jobs=1 because cross_validate_folds already runs the outer folds in parallel
    return GridSearchCV(pipeline, param_grid, cv=INNER_CV, scoring=SCORERS["rmse"], refit=True, n_jobs=1)

def score_folds(estimators: dict[str, BaseEstimator],
                *,
                X: pd.DataFrame,
                y: pd.Series,
                folds: dict[int, Folds]
) -> pd.DataFrame:
    # one row per (model, seed, fold), with that fold's unrounded scores from cross_validate_folds
    rows = []
    for seed, seed_folds in folds.items():
        for model_name, estimator in estimators.items():
            scores = cross_validate_folds(estimator, X=X, y=y, cv=seed_folds, scorers=SCORERS)
            for fold in range(len(seed_folds)):
                rows.append({
                    "model": model_name,
                    "seed": seed,
                    "fold": fold,
                    **{metric: fold_scores[fold] for metric, fold_scores in scores.items()},
                })
    return pd.DataFrame(rows)

def _format_value(value) -> str:
    # 3 significant digits, ex 0.0316 or 1e-05 or 1000
    if isinstance(value, (int, float, np.number)) and not isinstance(value, bool):
        return f"{float(f'{value:.3g}'):g}"
    return str(value)

def _is_edge(value, grid_values: list) -> bool:
    # a numeric value at the min or max of its grid, so the best value may lie past it
    if not all(isinstance(v, (int, float, np.number)) and not isinstance(v, bool) for v in grid_values):
        return False
    return value in (min(grid_values), max(grid_values))

def describe_setting(params: dict, param_grid: dict) -> dict:
    # ex {"setting": "alpha=0.0316, l1_ratio=0.5", "edge": True, "edge on": "l1_ratio"}
    at_edge = [name.removeprefix("model__") for name, value in params.items() if _is_edge(value, list(param_grid[name]))]
    return {
        "setting": ", ".join(f"{name.removeprefix('model__')}={_format_value(value)}" for name, value in params.items()),
        "edge": bool(at_edge),
        "edge on": ", ".join(at_edge),
    }

def chosen_settings(searches: dict[str, GridSearchCV],
                    fold_scores: pd.DataFrame,
                    *,
                    X: pd.DataFrame,
                    y: pd.Series,
                    folds: dict[int, Folds]
) -> pd.DataFrame:
    # cross_validate_folds keeps no fitted models, so each fold's search is refit here to read the setting it chose.
    # one row per (model, seed, fold)
    rows = []
    for seed, seed_folds in folds.items():
        for model_name, search in searches.items():
            for fold, (train, test) in enumerate(seed_folds):
                fitted = clone(search).fit(X.iloc[train], y.iloc[train])
                rows.append({
                    "model": model_name,
                    "seed": seed,
                    "fold": fold,
                    **describe_setting(fitted.best_params_, search.param_grid),
                    # the same scorer cross_validate uses, on the same validation rows
                    "rmse": -SCORERS["rmse"](fitted, X.iloc[test], y.iloc[test]),
                })

    chosen = pd.DataFrame(rows).merge(
        fold_scores[["model", "seed", "fold", "rmse"]],
        on=["model", "seed", "fold"],
        suffixes=("", "_cv"),
        validate="one_to_one",
    )
    # every fold has a cross_validate_folds score, and the refit search scores the same on it, so both are one procedure
    mismatch = chosen[~np.isclose(chosen["rmse"], chosen["rmse_cv"], rtol=0, atol=1e-10)]
    assert len(chosen) == len(rows), "a fold has no cross_validate_folds score"
    assert mismatch.empty, mismatch[["model", "seed", "fold", "setting", "rmse", "rmse_cv"]]
    return chosen

def score_folds_with_settings(searches: dict[str, GridSearchCV],
                              *,
                              X: pd.DataFrame,
                              y: pd.Series,
                              folds: dict[int, Folds]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    # score_folds and chosen_settings in one pass (K onward): cross_validate_folds also returns each fold's fitted
    # search, so the setting it chose is read from the model that was scored, with no refit.
    # returns the same scores frame as score_folds and a settings frame for settings_counts, one row per (model, seed, fold)
    score_rows, setting_rows = [], []
    for seed, seed_folds in folds.items():
        for model_name, search in searches.items():
            scores = cross_validate_folds(search, X=X, y=y, cv=seed_folds, scorers=SCORERS, return_estimator=True)
            fitted_searches = scores.pop("estimator")
            assert len(fitted_searches) == len(seed_folds), "a fold has no fitted search"
            for fold, fitted in enumerate(fitted_searches):
                fold_key = {"model": model_name, "seed": seed, "fold": fold}
                score_rows.append({
                    **fold_key,
                    **{metric: fold_scores[fold] for metric, fold_scores in scores.items()},
                })
                setting_rows.append({
                    **fold_key,
                    **describe_setting(fitted.best_params_, search.param_grid),
                    "rmse": scores["rmse"][fold],
                })
    return pd.DataFrame(score_rows), pd.DataFrame(setting_rows)

def settings_counts(chosen: pd.DataFrame) -> pd.DataFrame:
    # how often each setting was chosen across the outer folds: the models in their order, the most chosen setting first
    n_folds = len(chosen) // chosen["model"].nunique()
    counts = (
        chosen
        .groupby(["model", "setting"], sort=False)
        .agg(folds=("fold", "size"), edge=("edge", "first"), edge_on=("edge on", "first"))
        .reset_index()
    )
    counts["model"] = pd.Categorical(counts["model"], categories=chosen["model"].unique())
    return (
        counts
        .sort_values(["model", "folds"], ascending=[True, False], kind="stable")
        .set_index(["model", "setting"])
        .rename(columns={"folds": f"folds (of {n_folds})", "edge_on": "edge on"})
    )

def summary_table(fold_scores: pd.DataFrame, baseline_scores: pd.DataFrame) -> pd.DataFrame:
    # one row per model, the baseline first: the mean of each metric over the outer folds, the std of the seed means,
    # and the paired comparison with the baseline on the same folds
    assert not baseline_scores.duplicated(["seed", "fold"]).any(), "the baseline must be a single model"
    n_folds = len(baseline_scores)
    scores = pd.concat([baseline_scores.assign(model=baseline_scores["model"] + " (baseline)"), fold_scores], ignore_index=True)

    paired = scores.merge(
        baseline_scores[["seed", "fold", "rmse"]],
        on=["seed", "fold"],
        suffixes=("", "_baseline"),
        validate="many_to_one",
    )
    paired["gap"] = paired["rmse"] - paired["rmse_baseline"]
    # every model is paired with the baseline on all the outer folds
    assert (paired.groupby("model").size() == n_folds).all()

    summary = paired.groupby("model", sort=False).agg(
        rmse=("rmse", "mean"),
        mse=("mse", "mean"),
        mae=("mae", "mean"),
        gap=("gap", "mean"),
        wins=("gap", lambda gap: int((gap < -WIN_TOLERANCE).sum())),
    )
    summary.insert(3, "seed std", paired.groupby(["model", "seed"], sort=False)["rmse"].mean().groupby("model", sort=False).std())
    # the bar: better than the baseline by more than the noise floor
    summary["beats bar"] = summary["gap"] < -NOISE_FLOOR
    return summary.rename(columns={"wins": f"wins (of {n_folds})"})

def fit_final_models(searches: dict[str, GridSearchCV],
                     *,
                     X: pd.DataFrame,
                     y: pd.Series,
                     output_dir: Path
) -> tuple[pd.DataFrame, dict[str, Pipeline]]:
    # each search picks its setting on all the rows and refits it there; best_estimator_ is the pipeline that is saved
    rows, final_models = [], {}
    for model_name, search in searches.items():
        fitted = clone(search).fit(X, y)
        preprocessor = fitted.best_estimator_.named_steps["preprocess"]
        model_path = output_dir / model_file_name(model_name, preprocessing_name(preprocessor))
        joblib.dump(fitted.best_estimator_, model_path)

        final_models[model_name] = fitted.best_estimator_
        rows.append({
            "model": model_name,
            **describe_setting(fitted.best_params_, search.param_grid),
            "path": str(model_path),
        })
    return pd.DataFrame(rows).set_index("model"), final_models

# Fetch Data

> Only the training set is used here. The test set is held back so it plays no part in choosing the model.

In [4]:
train_df = read_csv_to_df(path=TRAIN_PATH)

X = train_df.drop(columns=TARGET)
y = train_df[TARGET]

# Baseline | Fixed Preprocessing, Untuned OLS

> the rows, preprocessing, and folds defined here are shared by J to M, and the untuned linear_regression scored here is the bar every tuned model has to beat

> **No saved model:** the baseline is F's linear_regression with Yeo-Johnson + StandardScaler, which F already saved, so it gets no output directory here

## Cleaning

> transforms and scalers don't fill missing values, so rows with missing values are dropped as in A, B, D, F, and I, leaving the 1,374 complete rows

In [5]:
def drop_missing_rows(X: pd.DataFrame, y: pd.Series) -> tuple[pd.DataFrame, pd.Series]:
    mask = X.notna().all(axis=1)
    return X[mask], y[mask]

X_clean, y_clean = drop_missing_rows(X, y)

## PreProcessing

| impute      | transform                                       | scale    |
|-------------|-------------------------------------------------|----------|
| passthrough | yeo-johnson (on all 6 skewed columns)           | standard |

> I's winner, with the same constructors as F and I: the PowerTransformer with `standardize=False` on `SKEWED` (|skew| ≥ 0.5 on the full training X), then `StandardScaler` on all 8 columns

In [6]:
# the skewed columns, using the same rule as Q1B: |skew| >= 0.5 on the training set
SKEW_THRESHOLD = 0.5
SKEWED = [c for c in X.columns if abs(X[c].skew()) >= SKEW_THRESHOLD]
assert len(SKEWED) == 6, SKEWED

# the one preprocessing every experiment in this notebook uses
preprocessor = create_preprocessing(preprocessing={
	"transform": on_columns(PowerTransformer(method="yeo-johnson", standardize=False), SKEWED),
	"scale": StandardScaler(),
})

## Folds

> I's 5 fold seeds on the complete rows, with no incomplete rows added (no imputer): 5 seeds x 4 folds = 20 outer folds. Seed 42 gives the folds of A to I, which the consistency check needs

In [7]:
# the fold seeds; keep SEED (42) in the list, since the consistency check compares its folds with F's
FOLD_SEEDS = [42, 0, 1, 2, 3]
FOLDS = {
	seed: complete_row_folds(X_clean, seed=seed, add_incomplete_to_train=False)
	for seed in FOLD_SEEDS
}
N_FOLDS = sum(len(seed_folds) for seed_folds in FOLDS.values())

# check: seed 42's folds are the folds of A to I (KFold on the complete rows)
folds_AtoI = list(KFold(n_splits=4, shuffle=True, random_state=SEED).split(X_clean))
assert len(FOLDS[SEED]) == len(folds_AtoI)
assert all(
    np.array_equal(train, train_AtoI) and np.array_equal(test, test_AtoI)
    for (train, test), (train_AtoI, test_AtoI) in zip(FOLDS[SEED], folds_AtoI)
)

print(f"{len(X_clean):,} complete rows, {len(FOLD_SEEDS)} seeds x {len(FOLDS[SEED])} folds = {N_FOLDS} outer folds, "
      f"training folds of {sorted({len(train) for seed_folds in FOLDS.values() for train, _ in seed_folds})} rows")

1,374 complete rows, 5 seeds x 4 folds = 20 outer folds, training folds of [1030, 1031] rows


## Cross-Validation Model Fitting

> no search: the untuned linear_regression is scored directly, with the same `score_folds` (and so the same `cross_validate_folds` call) as the tuned models

In [8]:
baseline_pipeline = build_pipelines(
    preprocessings=[preprocessor],
    models={"linear_regression": MODELS["linear_regression"]},
    seed=SEED,
)[("linear_regression", preprocessing_name(preprocessor))]

# one row per (seed, fold)
baseline_folds_df = score_folds({"linear_regression": baseline_pipeline}, X=X_clean, y=y_clean, folds=FOLDS)

## Consistency Check

> the baseline is F's and I's linear_regression with Yeo-Johnson + StandardScaler, so its mean RMSE on seed 42's folds, rounded to 3 decimals as `sort_results` does, must equal F's 0.527, and its mean over the 20 folds, rounded to 4 decimals, must equal I's 0.5258. That shows this notebook is on the same scale as F and I

In [9]:
baseline_seed_rmse = baseline_folds_df.loc[baseline_folds_df["seed"] == SEED, "rmse"].mean()
baseline_rmse = baseline_folds_df["rmse"].mean()
assert np.isclose(round(baseline_seed_rmse, 3), 0.527), baseline_seed_rmse
assert np.isclose(round(baseline_rmse, 4), 0.5258), baseline_rmse

print(f"baseline matches F on seed {SEED} ({baseline_seed_rmse:.3f}) and I over {N_FOLDS} folds ({baseline_rmse:.4f})")

baseline matches F on seed 42 (0.527) and I over 20 folds (0.5258)


# Experiment J | Regularization Strength

> **Hypothesis:** a tuned alpha brings Lasso and ElasticNet up to the OLS level (about 0.526), but no regularized model beats OLS by more than 0.002: with 8 features and about 1,030 training rows per fold there is little overfitting for a penalty to fix

> **Compare to:** the untuned linear_regression baseline on the same 20 folds (0.5258), and I's untuned ridge (0.5258), Lasso (1.1287), and ElasticNet (0.8582) on the same folds and preprocessing

> **Decision (decided before any results):** every J model goes to M as a candidate at its tuned setting, but it only counts as better than OLS if it beats the baseline by more than 0.002 in mean RMSE over the 20 folds

In [10]:
# the settings each model's search tries; edit a grid here to widen or refine it
experimentJ_grids = {
	"ridge": {"model__alpha": np.logspace(-3, 3, 13)},  # 0.001 to 1000, includes the default 1
	"lasso": {"model__alpha": np.logspace(-5, 0, 11)},  # 1e-5 to 1, includes the default 1
	"elastic_net": {
		"model__alpha": np.logspace(-5, 0, 11),  # 1e-5 to 1, includes the default 1
		# 0.01 and 0.05 added after a check run chose 0.1, the old low end, in 18 of 20 folds (near 0 it acts like ridge)
		"model__l1_ratio": [0.01, 0.05, 0.1, 0.5, 0.9],
	},
}

experimentJ_model_output_dir = MODEL_OUTPUT_DIR / 'J-REGULARIZATION-DropRowsWithMissingValues'
experimentJ_model_output_dir.mkdir(parents=True, exist_ok=True)

## Cleaning

> the shared rows and folds from the Baseline section: the 1,374 complete rows and the 20 outer folds, so in every fold each J model validates on the same rows as the baseline

In [11]:
X_experimentJ_clean, y_experimentJ_clean = X_clean, y_clean
experimentJ_folds = FOLDS

## PreProcessing

> the shared preprocessing, unchanged (Yeo-Johnson on the 6 skewed columns, then StandardScaler). Only the model settings vary:

| model       | setting            | grid                                                                    | candidates |
|-------------|--------------------|-------------------------------------------------------------------------|------------|
| ridge       | alpha              | 13 values, log-spaced from 0.001 to 1000 (includes the default 1)       | 13         |
| lasso       | alpha              | 11 values, log-spaced from 1e-5 to 1 (includes the default 1)           | 11         |
| elastic_net | alpha x l1_ratio   | the same 11 alphas as lasso x l1_ratio {0.01, 0.05, 0.1, 0.5, 0.9}      | 55         |

> every candidate is fit on 4 inner folds of each outer training fold, plus 1 refit of the chosen one, so ElasticNet runs 55 x 4 + 1 = 221 fits per outer fold

In [12]:
experimentJ_preProcessing = [preprocessor]

## Build Pipelines

> each pipeline is wrapped in its search (`nested_search`), and that search is what the outer folds score

In [13]:
# keyed by model name only, since the preprocessing is fixed; seeded as in the other notebooks
experimentJ_pipelines = {
	model_name: pipeline
	for (model_name, _), pipeline in build_pipelines(
		preprocessings=experimentJ_preProcessing,
		models={model_name: MODELS[model_name] for model_name in experimentJ_grids},
		seed=SEED,
	).items()
}
experimentJ_searches = {
	model_name: nested_search(pipeline, experimentJ_grids[model_name])
	for model_name, pipeline in experimentJ_pipelines.items()
}

## Cross-Validation Model Fitting

> the same `score_folds` call as the baseline, but the estimator is the search, so each outer fold runs its own inner search on its training rows only (nested CV, see Shared Setup)

In [14]:
# one row per (model, seed, fold)
experimentJ_folds_df = score_folds(experimentJ_searches, X=X_experimentJ_clean, y=y_experimentJ_clean, folds=experimentJ_folds)

## Results Tables

> **Summary vs the baseline:** one row per model, over the 20 paired outer folds. "gap" is the model's mean RMSE minus the baseline's (negative means better), "wins" counts the folds where the model's RMSE is lower by more than `WIN_TOLERANCE`, and "beats bar" is True only when the gap is below -0.002. The per-fold table is `experimentJ_folds_df`

In [15]:
experimentJ_summary = summary_table(experimentJ_folds_df, baseline_folds_df)
experimentJ_summary

,rmse,mse,mae,seed std,gap,wins (of 20),beats bar
model,,,,,,,
linear_regression (baseline),0.5258,0.2769,0.4084,0.0009,0.0000,0,False
ridge,0.5258,0.2769,0.4084,0.0008,-0.0000,12,False
lasso,0.5261,0.2772,0.4088,0.0011,0.0002,9,False
elastic_net,0.5258,0.2769,0.4085,0.0010,-0.0000,12,False


## Chosen Settings

> **Per outer fold:** each fold's search is refit on that fold's training rows to read the setting it chose, and the cell asserts it scores the same as `cross_validate_folds` on every fold. The table counts how often each setting was chosen across the 20 folds; "edge" is True when a chosen value is the min or max of its grid, and "edge on" names which. The per-fold table is `experimentJ_chosen_df`

In [16]:
experimentJ_chosen_df = chosen_settings(
    experimentJ_searches,
    experimentJ_folds_df,
    X=X_experimentJ_clean,
    y=y_experimentJ_clean,
    folds=experimentJ_folds,
)
print(f"all {len(experimentJ_chosen_df)} refit searches match their cross_validate_folds score")

experimentJ_settings = settings_counts(experimentJ_chosen_df)
experimentJ_settings

all 60 refit searches match their cross_validate_folds score


folds (of 20)   edge   edge on
model       setting                                                    
ridge       alpha=31.6                              13  False          
            alpha=10                                 7  False          
lasso       alpha=0.001                              6  False          
            alpha=0.000316                           4  False          
            alpha=0.01                               3  False          
            alpha=1e-05                              3   True     alpha
            alpha=0.00316                            2  False          
            alpha=3.16e-05                           1  False          
            alpha=0.0001                             1  False          
elastic_net alpha=0.0316, l1_ratio=0.01             13   True  l1_ratio
            alpha=0.0316, l1_ratio=0.05              2  False          
            alpha=0.01, l1_ratio=0.01                2   True  l1_ratio
            alpha=0.00316, l1_ratio=0.5              1  False          
            alpha=0.0316, l1_ratio=0.5               1  False          
            alpha=0.0316, l1_ratio=0.1               1  False

## Final Models

> each search picks its setting on all 1,374 complete rows and refits it there; the `best_estimator_` is saved, and its setting is recorded below

In [17]:
experimentJ_final_df, experimentJ_final_models = fit_final_models(
    experimentJ_searches,
    X=X_experimentJ_clean,
    y=y_experimentJ_clean,
    output_dir=experimentJ_model_output_dir,
)
experimentJ_final_df

,setting,edge,edge on,path
model,,,,
ridge,alpha=31.6,False,,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
lasso,alpha=0.000316,False,,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
elastic_net,"alpha=0.0316, l1_ratio=0.01",True,l1_ratio,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...


## Coefficients

> the final models' coefficients, next to the baseline's (also fit on all 1,374 rows). Every feature is standardized, so the coefficient sizes compare directly. The rows follow the pipeline's column order: the ColumnTransformer puts the 6 Yeo-Johnson columns first and the 2 passed-through ones after. "lasso is 0" marks the features Lasso drops (a coefficient of exactly 0)

In [18]:
# the baseline refit on all the rows too, so its coefficients sit next to the tuned models'
experimentJ_coef_models = {
	"linear_regression": clone(baseline_pipeline).fit(X_experimentJ_clean, y_experimentJ_clean),
	**experimentJ_final_models,
}

# names from the fitted preprocessing, ex "transform__Generosity" (a Yeo-Johnson column) or "remainder__Social support"
experimentJ_feature_names = experimentJ_coef_models["linear_regression"].named_steps["preprocess"].get_feature_names_out()
assert all(
    np.array_equal(pipeline.named_steps["preprocess"].get_feature_names_out(), experimentJ_feature_names)
    for pipeline in experimentJ_coef_models.values()
)

experimentJ_coefs = pd.DataFrame(
    {model_name: pipeline.named_steps["model"].coef_ for model_name, pipeline in experimentJ_coef_models.items()},
    index=pd.Index([SHORT_NAMES.get(name.split("__", 1)[1], name) for name in experimentJ_feature_names], name="feature"),
)
experimentJ_coefs.insert(0, "yeo-johnson", [name.startswith("transform__") for name in experimentJ_feature_names])
experimentJ_coefs["lasso is 0"] = experimentJ_coefs["lasso"] == 0

experimentJ_lasso_zeros = list(experimentJ_coefs.index[experimentJ_coefs["lasso is 0"]])
print(f"Lasso sets {len(experimentJ_lasso_zeros)} of {len(experimentJ_coefs)} coefficients to exactly 0: {experimentJ_lasso_zeros or 'none'}")
experimentJ_coefs

Lasso sets 0 of 8 coefficients to exactly 0: none


,yeo-johnson,linear_regression,ridge,lasso,elastic_net,lasso is 0
feature,,,,,,
Social,True,0.2813,0.2772,0.2809,0.2755,False
Life exp,True,0.3059,0.3045,0.3061,0.3040,False
Freedom,True,0.0351,0.0426,0.0352,0.0451,False
Generosity,True,0.0775,0.0755,0.0773,0.0746,False
Corruption,True,-0.1097,-0.1073,-0.1094,-0.1062,False
Neg affect,True,0.0299,0.0258,0.0293,0.0239,False
GDP,False,0.3245,0.3199,0.3243,0.3181,False
Pos affect,False,0.2059,0.2004,0.2057,0.1984,False


## Experiment J Results

- **Hypothesis holds:** tuning alpha brings Lasso (1.1287 untuned in I → 0.5261) and ElasticNet (0.8582 → 0.5258) up to the OLS level, but no regularized model beats the 0.5258 baseline by more than 0.002. Ridge and ElasticNet tie it within 0.0001 and win 12 of 20 folds (a coin flip); Lasso is 0.0002 behind and wins 9.
- **Ridge prefers a stronger penalty, but it makes no difference:** every fold chose alpha 10 or 31.6 (31.6 in 13 folds and in the final fit), well above the default 1, yet RMSE is unchanged and the four largest coefficients shrink by only 0.5 to 3% (ex GDP 0.3245 → 0.3199). With about 1,030 training rows, a penalty of this size barely moves the fit.
- **Lasso wants almost no penalty and drops no features:** its chosen alpha spreads over 7 values from 1e-5 to 0.01 (3 folds at the 1e-5 low edge), the final fit uses 0.000316, and 0 of 8 coefficients are 0. Every feature carries some signal.
- **ElasticNet slides toward ridge:** l1_ratio=0.01, the new low edge, was chosen in 15 of 20 folds (0.05 in 2 more) and in the final fit (alpha 0.0316). As l1_ratio approaches 0 ElasticNet becomes ridge, which J already tunes directly, so widening the grid again would not change the result.
- **All four models agree on the coefficients** (standardized features): GDP 0.32, Life exp 0.31, Social 0.28, and Pos affect 0.21 carry most of the weight, Corruption is -0.11, and Generosity (0.08), Freedom (0.04), and Neg affect (0.03) are small.
- **Carry forward**
  - **For M:** ridge (alpha 31.6), Lasso (alpha 0.000316), and ElasticNet (alpha 0.0316, l1_ratio 0.01) all tie linear_regression, so under the simplicity order linear_regression stays ahead of all three.
  - **What this says about the model:** regularization can't help because the linear model is not overfitting. Its error comes from bias (what a linear fit on these 8 features can capture), not from variance, so K and L are not expected to beat 0.5258 either.
  - **For K:** the bar stays 0.5258 over the same 20 folds. The edge flags from J point toward "no penalty" or "ridge", both already covered, so they need no follow-up.

# Experiment K | SVR Margin Settings

> **Hypothesis:** tuning C and epsilon closes svr_linear's and LinearSVR's 0.0025 to 0.0035 RMSE gap to OLS (I: svr_linear 0.5283, LinearSVR 0.5293, baseline 0.5258) while keeping their MAE lead: both had the lowest MAE in F, and the baseline's MAE over the 20 folds is about 0.4084

> **Compare to:** the untuned linear_regression baseline on the same 20 folds (0.5258), I's untuned svr_linear (0.5283) and LinearSVR (0.5293) on the same folds and preprocessing, and J's tuned ridge, Lasso, and ElasticNet

> **Decision (decided before any results):** both K models go to M as candidates at their tuned settings, but a K model only counts as better than OLS if it beats the baseline by more than 0.002 in mean RMSE over the 20 folds. Since the hypothesis is about MAE too, the MAE gap vs the baseline (model minus baseline, negative means better) and the MAE wins of 20 are reported next to it

In [19]:
# the settings each model's search tries; edit a grid here to widen or refine it
experimentK_grids = {
	"svr_linear": {
		"model__C": [0.01, 0.1, 1, 10, 100],  # includes the default 1
		"model__epsilon": [0, 0.1, 0.25, 0.5],  # on the target's scale, includes SVR's default 0.1
	},
	"linear_svr": {
		"model__C": [0.01, 0.1, 1, 10, 100],  # includes the default 1
		"model__epsilon": [0, 0.1, 0.25, 0.5],  # on the target's scale, includes LinearSVR's default 0
	},
}

experimentK_model_output_dir = MODEL_OUTPUT_DIR / 'K-SVR-MARGIN-DropRowsWithMissingValues'
experimentK_model_output_dir.mkdir(parents=True, exist_ok=True)

## Cleaning

> the shared rows and folds from the Baseline section: the 1,374 complete rows and the 20 outer folds, so in every fold each K model validates on the same rows as the baseline and the J models

In [20]:
X_experimentK_clean, y_experimentK_clean = X_clean, y_clean
experimentK_folds = FOLDS

## PreProcessing

> the shared preprocessing, unchanged (Yeo-Johnson on the 6 skewed columns, then StandardScaler). Only the model settings vary:

| model      | setting     | grid                                                       | candidates |
|------------|-------------|------------------------------------------------------------|------------|
| svr_linear | C x epsilon | C {0.01, 0.1, 1, 10, 100} x epsilon {0, 0.1, 0.25, 0.5}    | 20         |
| linear_svr | C x epsilon | the same 5 Cs x 4 epsilons as svr_linear                   | 20         |

> epsilon is the half-width of the tube where errors cost nothing, so it is on the target's scale (the target's std is 1.13), not the standardized features'. Both defaults are inside the grid: C=1 for both, epsilon 0.1 for SVR and 0 for LinearSVR

> every candidate is fit on 4 inner folds of each outer training fold, plus 1 refit of the chosen one, so each model runs 20 x 4 + 1 = 81 fits per outer fold

In [21]:
experimentK_preProcessing = [preprocessor]

## Build Pipelines

> each pipeline is wrapped in its search (`nested_search`), and that search is what the outer folds score. The models keep their `MODELS` settings: SVR with `kernel="linear"`, `max_iter=1_000_000` and LinearSVR with `dual=True`, `max_iter=10_000`

In [22]:
# keyed by model name only, since the preprocessing is fixed; seeded as in the other notebooks
experimentK_pipelines = {
	model_name: pipeline
	for (model_name, _), pipeline in build_pipelines(
		preprocessings=experimentK_preProcessing,
		models={model_name: MODELS[model_name] for model_name in experimentK_grids},
		seed=SEED,
	).items()
}
experimentK_searches = {
	model_name: nested_search(pipeline, experimentK_grids[model_name])
	for model_name, pipeline in experimentK_pipelines.items()
}

## Cross-Validation Model Fitting

> the same nested CV as J, scored with `score_folds_with_settings`: it makes the same `cross_validate_folds` call as `score_folds`, and also keeps each fold's fitted search, so the setting each fold chose is read from it with no refit (see Shared Setup)

In [23]:
# one row per (model, seed, fold) in both: the scores, and the setting that fold's search chose
experimentK_folds_df, experimentK_chosen_df = score_folds_with_settings(
    experimentK_searches,
    X=X_experimentK_clean,
    y=y_experimentK_clean,
    folds=experimentK_folds,
)

/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000000).  Consider pre-processing your data with StandardScaler or Min

## Results Tables

> **Summary vs the baseline:** one row per model, over the 20 paired outer folds. "gap" is the model's mean RMSE minus the baseline's (negative means better), "wins" counts the folds where the model's RMSE is lower by more than `WIN_TOLERANCE`, and "beats bar" is True only when the gap is below -0.002. "mae gap" and "mae wins" are the same paired comparison on MAE, since the hypothesis is about MAE too; they don't change "beats bar". The per-fold table is `experimentK_folds_df`

In [24]:
experimentK_summary = summary_table(experimentK_folds_df, baseline_folds_df)

# the hypothesis is about MAE too, so the same paired comparison with the baseline as "gap" and "wins", on MAE
experimentK_mae_paired = experimentK_folds_df.merge(
    baseline_folds_df[["seed", "fold", "mae"]],
    on=["seed", "fold"],
    suffixes=("", "_baseline"),
    validate="many_to_one",
)
experimentK_mae_paired["mae_gap"] = experimentK_mae_paired["mae"] - experimentK_mae_paired["mae_baseline"]
# every model is paired with the baseline on all the outer folds
assert (experimentK_mae_paired.groupby("model").size() == N_FOLDS).all()

experimentK_mae = experimentK_mae_paired.groupby("model", sort=False).agg(
    mae_gap=("mae_gap", "mean"),
    mae_wins=("mae_gap", lambda gap: int((gap < -WIN_TOLERANCE).sum())),
)
# the baseline against itself: a gap of 0 and no wins, as in summary_table
experimentK_summary = experimentK_summary.join(
    experimentK_mae
    .reindex(experimentK_summary.index, fill_value=0)
    .rename(columns={"mae_gap": "mae gap", "mae_wins": f"mae wins (of {N_FOLDS})"})
)
experimentK_summary

,rmse,mse,mae,seed std,gap,wins (of 20),beats bar,mae gap,mae wins (of 20)
model,,,,,,,,,
linear_regression (baseline),0.5258,0.2769,0.4084,0.0009,0.0000,0,False,0.0000,0
svr_linear,0.5278,0.2791,0.4082,0.0014,0.0020,4,False,-0.0002,11
linear_svr,0.5274,0.2786,0.4076,0.0011,0.0016,6,False,-0.0008,13


## Chosen Settings

> **Per outer fold:** the setting each fold's search chose, read from the fitted search that `cross_validate_folds` returned for that fold (the same model that was scored), so nothing is refit. The table counts how often each setting was chosen across the 20 folds; "edge" is True when a chosen value is the min or max of its grid, and "edge on" names which. epsilon=0 is flagged as an edge, but epsilon can't go below 0, so that edge needs no widening. The per-fold table is `experimentK_chosen_df`

In [25]:
experimentK_settings = settings_counts(experimentK_chosen_df)
experimentK_settings

folds (of 20)   edge     edge on
model      setting                                               
svr_linear C=100, epsilon=0.25               4   True           C
           C=10, epsilon=0.25                4  False            
           C=0.1, epsilon=0.25               3  False            
           C=1, epsilon=0.5                  2   True     epsilon
           C=0.1, epsilon=0.5                2   True     epsilon
           C=10, epsilon=0.5                 2   True     epsilon
           C=0.01, epsilon=0                 1   True  C, epsilon
           C=0.01, epsilon=0.25              1   True           C
           C=100, epsilon=0.5                1   True  C, epsilon
linear_svr C=10, epsilon=0.25                6  False            
           C=10, epsilon=0.5                 5   True     epsilon
           C=1, epsilon=0.25                 4  False            
           C=10, epsilon=0                   1   True     epsilon
           C=0.1, epsilon=0.1                1  False            
           C=1, epsilon=0.1                  1  False            
           C=1, epsilon=0.5                  1   True     epsilon
           C=0.1, epsilon=0.25               1  False

## Final Models

> each search picks its setting on all 1,374 complete rows and refits it there; the `best_estimator_` is saved, and its setting is recorded below

In [26]:
experimentK_final_df, experimentK_final_models = fit_final_models(
    experimentK_searches,
    X=X_experimentK_clean,
    y=y_experimentK_clean,
    output_dir=experimentK_model_output_dir,
)
experimentK_final_df

/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/opt/homebrew/Caskroom/miniconda/base/envs/homl3/lib/python3.10/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000000).  Consider pre-processing your data with StandardScaler or Min

,setting,edge,edge on,path
model,,,,
svr_linear,"C=0.01, epsilon=0.1",True,C,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...
linear_svr,"C=0.1, epsilon=0.25",False,,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...


## Experiment K Results

- **Hypothesis partly holds:** tuning narrows the RMSE gap to OLS (svr_linear 0.0025 → 0.0020, LinearSVR 0.0035 → 0.0016), so both now tie the 0.5258 baseline within the 0.002 noise floor (svr_linear 0.5278, LinearSVR 0.5274). But neither beats it, and they still lose most folds: svr_linear wins 4 of 20, LinearSVR 6 of 20.
- **The MAE lead is kept, but it is small:** MAE gap -0.0002 for svr_linear (11 of 20 folds) and -0.0008 for LinearSVR (13 of 20, MAE 0.4076 vs 0.4084). The SVR models still trade a little RMSE for a little MAE, as expected from their absolute-style loss.
- **Both models want a wider margin than their defaults:** epsilon 0.25 or 0.5 was chosen in 19 of 20 folds for svr_linear and 17 of 20 for LinearSVR (defaults are 0.1 and 0). Ignoring residuals smaller than about 0.25 (the target's std is 1.13) costs nothing here.
- **C barely matters:** svr_linear's choice spreads over 9 settings with C from 0.01 to 100, and its final all-rows fit picked C=0.01, the low edge, while 5 folds picked C=100, the high edge. LinearSVR mostly picked C=10 (12 folds), but its final fit picked C=0.1. A setting that jumps across 4 orders of magnitude means the score surface is flat.
- **Large C did not converge:** the run logged 615 "Liblinear failed to converge" and 296 "Solver terminated early" warnings during cross-validation (plus 29 and 16 in the final fits). Each model ran 1,620 fits (20 outer folds x 81), and the counts roughly match the large-C share of the grid: 615 is 38% of LinearSVR's fits (C=10 and 100 are 8 of 20 candidates, 40%), and 296 is 18% of svr_linear's (C=100 is 4 of 20, 20%). So the folds that chose a large C most likely scored a truncated fit. Both final models use small C (0.01 and 0.1), which are likely unaffected.
- **Carry forward**
  - **For M:** svr_linear (C=0.01, epsilon=0.1) and LinearSVR (C=0.1, epsilon=0.25) tie OLS on RMSE but rank last in the simplicity order, so linear_regression stays ahead.
  - **Optional check:** raising max_iter for the large-C fits would remove the warnings, but with a flat score surface and J's finding that the linear model's error comes from bias, it is unlikely to change the decision.
  - **For L:** the bar stays 0.5258. SGD is the last model to tune before M.

# Experiment L | SGD

> **Hypothesis:** SGD's 0.0003 RMSE gap to OLS (I: SGD 0.5261, baseline 0.5258) comes from its optimizer, not its settings: stochastic updates stop near the least-squares solution, not at it. So tuning brings SGD level with OLS but not past it

> **Compare to:** the untuned linear_regression baseline on the same 20 folds (0.5258), I's untuned SGD (0.5261) on the same folds and preprocessing, and J's and K's tuned models

> **Decision (decided before any results):** SGD goes to M as a candidate at its tuned settings, but it only counts as better than OLS if it beats the baseline by more than 0.002 in mean RMSE over the 20 folds. The MAE gap and MAE wins are reported next to it as in K, and the final SGD's coefficients and intercept are compared with the baseline's (both fit on all 1,374 rows) to test the hypothesis

In [27]:
# the settings the search tries; edit the grid here to widen or refine it
experimentL_grids = {
	"sgd": {
		# 0.1 added after a check run chose 0.01, the old high end, in 17 of 20 folds (J's ridge alpha 31.6 / about 1,030 rows is about 0.03)
		"model__alpha": [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1],  # the penalty's strength, includes the default 1e-4
		"model__penalty": ["l2", "l1"],  # includes the default l2
		"model__learning_rate": ["invscaling", "adaptive"],  # includes the default invscaling; eta0 stays at its default 0.01
	},
}

experimentL_model_output_dir = MODEL_OUTPUT_DIR / 'L-SGD-DropRowsWithMissingValues'
experimentL_model_output_dir.mkdir(parents=True, exist_ok=True)

## Cleaning

> the shared rows and folds from the Baseline section: the 1,374 complete rows and the 20 outer folds, so in every fold SGD validates on the same rows as the baseline and the J and K models

In [28]:
X_experimentL_clean, y_experimentL_clean = X_clean, y_clean
experimentL_folds = FOLDS

## PreProcessing

> the shared preprocessing, unchanged (Yeo-Johnson on the 6 skewed columns, then StandardScaler). Only the model settings vary:

| model | setting                         | grid                                                                                           | candidates |
|-------|---------------------------------|------------------------------------------------------------------------------------------------|------------|
| sgd   | alpha x penalty x learning_rate | alpha {1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1} x penalty {l2, l1} x learning_rate {invscaling, adaptive} | 24         |

> alpha multiplies the penalty term. All three defaults are inside the grid: alpha 1e-4, penalty l2, learning_rate invscaling

> eta0, the starting learning rate, stays at its default 0.01 for both schedules: invscaling shrinks it on every update (eta0 / t^0.25), and adaptive keeps it constant and divides it by 5 each time the training loss stops improving (by less than `tol` for 5 epochs in a row)

> every candidate is fit on 4 inner folds of each outer training fold, plus 1 refit of the chosen one, so SGD runs 24 x 4 + 1 = 97 fits per outer fold

In [29]:
experimentL_preProcessing = [preprocessor]

## Build Pipelines

> the pipeline is wrapped in its search (`nested_search`), and that search is what the outer folds score. The model keeps its `MODELS` settings, SGDRegressor's defaults (squared-error loss, `max_iter=1000`, `tol=1e-3`), with `random_state` set to `SEED` by `build_pipelines`, so every fit shuffles the rows the same way. A fit stops once the training loss fails to improve by `tol` for 5 epochs in a row (with adaptive, only after its learning rate has fallen below 1e-6), or at `max_iter`, which raises a ConvergenceWarning

In [30]:
# keyed by model name only, since the preprocessing is fixed; seeded as in the other notebooks
experimentL_pipelines = {
	model_name: pipeline
	for (model_name, _), pipeline in build_pipelines(
		preprocessings=experimentL_preProcessing,
		models={model_name: MODELS[model_name] for model_name in experimentL_grids},
		seed=SEED,
	).items()
}
experimentL_searches = {
	model_name: nested_search(pipeline, experimentL_grids[model_name])
	for model_name, pipeline in experimentL_pipelines.items()
}

## Cross-Validation Model Fitting

> the same nested CV as J and K, scored with `score_folds_with_settings` as in K: the same `cross_validate_folds` call as `score_folds`, keeping each fold's fitted search so the setting each fold chose is read from it with no refit (see Shared Setup)

In [31]:
# one row per (model, seed, fold) in both: the scores, and the setting that fold's search chose
experimentL_folds_df, experimentL_chosen_df = score_folds_with_settings(
    experimentL_searches,
    X=X_experimentL_clean,
    y=y_experimentL_clean,
    folds=experimentL_folds,
)

## Results Tables

> **Summary vs the baseline:** one row per model, over the 20 paired outer folds. "gap" is the model's mean RMSE minus the baseline's (negative means better), "wins" counts the folds where the model's RMSE is lower by more than `WIN_TOLERANCE`, and "beats bar" is True only when the gap is below -0.002. "mae gap" and "mae wins" are the same paired comparison on MAE, as in K, so SGD's MAE compares with the K models'; they don't change "beats bar". The per-fold table is `experimentL_folds_df`

In [32]:
experimentL_summary = summary_table(experimentL_folds_df, baseline_folds_df)

# the same paired comparison with the baseline as "gap" and "wins", on MAE, as in K
experimentL_mae_paired = experimentL_folds_df.merge(
    baseline_folds_df[["seed", "fold", "mae"]],
    on=["seed", "fold"],
    suffixes=("", "_baseline"),
    validate="many_to_one",
)
experimentL_mae_paired["mae_gap"] = experimentL_mae_paired["mae"] - experimentL_mae_paired["mae_baseline"]
# every model is paired with the baseline on all the outer folds
assert (experimentL_mae_paired.groupby("model").size() == N_FOLDS).all()

experimentL_mae = experimentL_mae_paired.groupby("model", sort=False).agg(
    mae_gap=("mae_gap", "mean"),
    mae_wins=("mae_gap", lambda gap: int((gap < -WIN_TOLERANCE).sum())),
)
# the baseline against itself: a gap of 0 and no wins, as in summary_table
experimentL_summary = experimentL_summary.join(
    experimentL_mae
    .reindex(experimentL_summary.index, fill_value=0)
    .rename(columns={"mae_gap": "mae gap", "mae_wins": f"mae wins (of {N_FOLDS})"})
)
experimentL_summary

,rmse,mse,mae,seed std,gap,wins (of 20),beats bar,mae gap,mae wins (of 20)
model,,,,,,,,,
linear_regression (baseline),0.5258,0.2769,0.4084,0.0009,0.0000,0,False,0.0000,0
sgd,0.5261,0.2772,0.4088,0.0011,0.0003,7,False,0.0004,9


## Chosen Settings

> **Per outer fold:** the setting each fold's search chose, read from the fitted search that `cross_validate_folds` returned for that fold (the same model that was scored), so nothing is refit. The table counts how often each setting was chosen across the 20 folds; "edge" is True when a chosen value is the min or max of its grid, and "edge on" names which. Only alpha is numeric, so only alpha can be flagged: penalty and learning_rate are choices, and the grid already holds both values of each. The per-fold table is `experimentL_chosen_df`

In [33]:
experimentL_settings = settings_counts(experimentL_chosen_df)
experimentL_settings

folds (of 20)   edge  \
model setting                                                                   
sgd   alpha=0.01, learning_rate=adaptive, penalty=l2                10  False   
      alpha=0.01, learning_rate=invscaling, penalty=l2               4  False   
      alpha=0.01, learning_rate=adaptive, penalty=l1                 2  False   
      alpha=0.001, learning_rate=invscaling, penalty=l1              1  False   
      alpha=0.01, learning_rate=invscaling, penalty=l1               1  False   
      alpha=0.001, learning_rate=invscaling, penalty=l2              1  False   
      alpha=0.001, learning_rate=adaptive, penalty=l1                1  False   

                                                        edge on  
model setting                                                    
sgd   alpha=0.01, learning_rate=adaptive, penalty=l2             
      alpha=0.01, learning_rate=invscaling, penalty=l2           
      alpha=0.01, learning_rate=adaptive, penalty=l1             
      alpha=0.001, learning_rate=invscaling, penalty=l1          
      alpha=0.01, learning_rate=invscaling, penalty=l1           
      alpha=0.001, learning_rate=invscaling, penalty=l2          
      alpha=0.001, learning_rate=adaptive, penalty=l1

## Final Models

> the search picks its setting on all 1,374 complete rows and refits it there; the `best_estimator_` is saved, and its setting is recorded below

In [34]:
experimentL_final_df, experimentL_final_models = fit_final_models(
    experimentL_searches,
    X=X_experimentL_clean,
    y=y_experimentL_clean,
    output_dir=experimentL_model_output_dir,
)
experimentL_final_df

,setting,edge,edge on,path
model,,,,
sgd,"alpha=0.01, learning_rate=invscaling, penalty=l2",False,,/Users/mrt/Documents/MrT/IUBloomington/Fall 20...


## Coefficients

> the final SGD's coefficients and intercept, next to the baseline linear_regression's, both fit on all 1,374 complete rows. The baseline is the refit from J's Coefficients cell (`experimentJ_coef_models["linear_regression"]`), so nothing is refit here. The features are named and ordered as in J's table (the 6 Yeo-Johnson columns first). "difference" is SGD minus the baseline: if SGD stops near the least-squares solution, not at it, the two coefficient sets differ by a little

In [35]:
# the baseline refit on all the rows, from J's Coefficients cell, and the final SGD, fit on the same rows
experimentL_coef_models = {
	"linear_regression": experimentJ_coef_models["linear_regression"],
	**experimentL_final_models,
}

# the same fitted preprocessing, so the same feature names as J's table
experimentL_feature_names = experimentL_coef_models["sgd"].named_steps["preprocess"].get_feature_names_out()
assert np.array_equal(experimentL_feature_names, experimentJ_feature_names)

# the coefficients, then the intercept (SGDRegressor stores it as a 1-element array, LinearRegression as a float)
experimentL_coefs = pd.DataFrame(
	{
		model_name: [*pipeline.named_steps["model"].coef_, float(np.ravel(pipeline.named_steps["model"].intercept_)[0])]
		for model_name, pipeline in experimentL_coef_models.items()
	},
	index=pd.Index([SHORT_NAMES.get(name.split("__", 1)[1], name) for name in experimentL_feature_names] + ["intercept"], name="feature"),
)
experimentL_coefs["difference"] = experimentL_coefs["sgd"] - experimentL_coefs["linear_regression"]

experimentL_coef_diffs = experimentL_coefs["difference"].drop("intercept").abs()
print(f"max |difference| over the {len(experimentL_coef_diffs)} coefficients: {experimentL_coef_diffs.max():.4f} "
      f"({experimentL_coef_diffs.idxmax()}); intercept difference: {experimentL_coefs.loc['intercept', 'difference']:.4f}")
experimentL_coefs

max |difference| over the 8 coefficients: 0.0058 (GDP); intercept difference: 0.0027


,linear_regression,sgd,difference
feature,,,
Social,0.2813,0.2781,-0.0032
Life exp,0.3059,0.3084,0.0024
Freedom,0.0351,0.0343,-0.0009
Generosity,0.0775,0.0738,-0.0038
Corruption,-0.1097,-0.1072,0.0025
Neg affect,0.0299,0.0311,0.0012
GDP,0.3245,0.3186,-0.0058
Pos affect,0.2059,0.2014,-0.0046
intercept,5.4525,5.4552,0.0027


## Experiment L Results

- **Hypothesis holds:** tuned SGD scores 0.5261, exactly its untuned score in I, so tuning did not move it. It ties the 0.5258 baseline within the noise floor (gap 0.0003, wins 7 of 20 folds) and does not beat it. Its MAE is also slightly worse (0.4088 vs 0.4084, MAE wins 9 of 20).
- **SGD wants a stronger penalty than its default, like ridge:** alpha 0.01 was chosen in 17 of 20 folds (0.001 in the other 3), 100 times the default 1e-4. SGD's alpha is roughly ridge's alpha divided by the number of rows, so 0.01 x about 1,030 rows is about 10, in line with J's ridge choice of 10 to 31.6.
- **The wider grid confirmed it:** after a check run chose 0.01 at the old top of the grid, alpha 0.1 was added. No fold chose 0.1, so 0.01 is now an interior choice, not a grid edge.
- **Penalty type and schedule matter less:** l2 in 15 of 20 folds, adaptive learning rate in 13 of 20. The final all-rows fit uses alpha 0.01, l2, invscaling: SGD's defaults with a stronger penalty. No convergence warnings in any fit.
- **SGD lands next to OLS:** every final coefficient is within 0.0058 of the baseline's (largest gap on GDP) and the intercept within 0.0027. GDP, Pos affect, Social, and Generosity shrink slightly, the same direction ridge shrank them in J, so part of the gap is the l2 penalty and part is where SGD stops. This table can't separate the two.
- **Carry forward**
  - **For M:** SGD (alpha 0.01, l2, invscaling) ties OLS on RMSE but sits below linear_regression, ridge, Lasso, and ElasticNet in the simplicity order.
  - **All models are now tuned:** across J, K, and L every model is within 0.002 of the 0.5258 baseline (svr_linear right at the edge) (ridge and ElasticNet 0.5258, Lasso and SGD 0.5261, LinearSVR 0.5274, svr_linear 0.5278), and none clears the bar. Under M's rule (decided before any results), linear_regression with the fixed preprocessing is the pick.

# Experiment M | Final Selection

> **Purpose:** pick the final pipeline from the untuned baseline and every tuned model in J, K, and L, with the rule stated in Shared Setup before any results. M runs no new tuning and no new cross-validation: it only reads the scores, settings, and final models of the baseline and of J, K, and L. The test set is not used here; it is scored once, in a later step

> **Rule (from Shared Setup, decided before any results):** the best mean RMSE over the 20 folds wins; if the top models are within 0.002, take the simplest in this order:
> 1. linear_regression
> 2. ridge
> 3. Lasso / ElasticNet
> 4. SGD
> 5. svr_linear / LinearSVR

> "within 0.002" means a mean RMSE no more than the best mean RMSE + 0.002 (`NOISE_FLOOR`). Within the same rank (Lasso vs ElasticNet, svr_linear vs LinearSVR), the lower mean RMSE wins

> **Decision:** the rule picks; the chosen pipeline is saved for the test-set step

In [36]:
# the simplicity order from Shared Setup (decided before any results): a lower rank is simpler
experimentM_simplicity_rank = {
	"linear_regression": 1,
	"ridge": 2,
	"lasso": 3,
	"elastic_net": 3,
	"sgd": 4,
	"svr_linear": 5,
	"linear_svr": 5,
}

# what M reads from each experiment: its per-fold scores, its summary table, its final settings, and its final models
experimentM_sources = {
	"J": (experimentJ_folds_df, experimentJ_summary, experimentJ_final_df, experimentJ_final_models),
	"K": (experimentK_folds_df, experimentK_summary, experimentK_final_df, experimentK_final_models),
	"L": (experimentL_folds_df, experimentL_summary, experimentL_final_df, experimentL_final_models),
}

experimentM_model_output_dir = MODEL_OUTPUT_DIR / 'M-FINAL-DropRowsWithMissingValues'
experimentM_model_output_dir.mkdir(parents=True, exist_ok=True)

## Candidates

> one row per candidate: the untuned baseline (linear_regression) and each tuned model from J, K, and L. The scores are the 20 outer folds those experiments already scored, put in one frame and passed through the same `summary_table`, so every number is the one in J's, K's, or L's table (the cell asserts it, row by row). "final setting" is the setting from that experiment's Final Models table, "gap", "wins", and "beats bar" are vs the baseline as before, "simplicity rank" is the order above, and "tied with best" is True when the mean RMSE is no more than the best mean RMSE + 0.002. Sorted by simplicity rank, then mean RMSE

In [37]:
# every tuned model's outer folds in one frame; summary_table adds the baseline, as in J, K, and L
experimentM_folds_df = pd.concat([folds_df for folds_df, _, _, _ in experimentM_sources.values()], ignore_index=True)
assert not experimentM_folds_df.duplicated(["model", "seed", "fold"]).any(), "a model is in more than one experiment"

experimentM_candidates = summary_table(experimentM_folds_df, baseline_folds_df)

# M's rows (the baseline's included) must equal the rows of J's, K's, and L's tables, so M picks from the numbers reported there
experimentM_shared_columns = ["rmse", "mse", "mae", "seed std", "gap", f"wins (of {N_FOLDS})", "beats bar"]
for experiment, (_, summary, _, _) in experimentM_sources.items():
	pd.testing.assert_frame_equal(
		experimentM_candidates.loc[summary.index, experimentM_shared_columns],
		summary[experimentM_shared_columns],
		check_exact=True,
		obj=f"Experiment {experiment}'s summary",
	)
print(f"every row matches its row in {', '.join(f'experiment{experiment}_summary' for experiment in experimentM_sources)}")

# summary_table names the baseline "linear_regression (baseline)"; it is the untuned linear_regression, so M keys it by that model name
experimentM_baseline_name = baseline_folds_df["model"].iloc[0]
experimentM_candidates = experimentM_candidates.rename(index={f"{experimentM_baseline_name} (baseline)": experimentM_baseline_name})

# where each candidate comes from, and its final setting from that experiment's Final Models table
experimentM_origin = {experimentM_baseline_name: ("baseline", "none (untuned)")}
for experiment, (_, _, final_df, _) in experimentM_sources.items():
	experimentM_origin.update({model_name: (experiment, setting) for model_name, setting in final_df["setting"].items()})
assert set(experimentM_origin) == set(experimentM_candidates.index) == set(experimentM_simplicity_rank)

experimentM_candidates.insert(0, "experiment", experimentM_candidates.index.map(lambda model_name: experimentM_origin[model_name][0]))
experimentM_candidates.insert(1, "final setting", experimentM_candidates.index.map(lambda model_name: experimentM_origin[model_name][1]))
experimentM_candidates["simplicity rank"] = experimentM_candidates.index.map(experimentM_simplicity_rank)

# tied with the best: a mean RMSE within the noise floor of the best one
experimentM_best_rmse = experimentM_candidates["rmse"].min()
experimentM_candidates["tied with best"] = experimentM_candidates["rmse"] <= experimentM_best_rmse + NOISE_FLOOR

experimentM_candidates = experimentM_candidates.sort_values(["simplicity rank", "rmse"], kind="stable")
experimentM_candidates

every row matches its row in experimentJ_summary, experimentK_summary, experimentL_summary


,experiment,final setting,rmse,mse,mae,seed std,gap,wins (of 20),beats bar,simplicity rank,tied with best
model,,,,,,,,,,,
linear_regression,baseline,none (untuned),0.5258,0.2769,0.4084,0.0009,0.0000,0,False,1,True
ridge,J,alpha=31.6,0.5258,0.2769,0.4084,0.0008,-0.0000,12,False,2,True
elastic_net,J,"alpha=0.0316, l1_ratio=0.01",0.5258,0.2769,0.4085,0.0010,-0.0000,12,False,3,True
lasso,J,alpha=0.000316,0.5261,0.2772,0.4088,0.0011,0.0002,9,False,3,True
sgd,L,"alpha=0.01, learning_rate=invscaling, penalty=l2",0.5261,0.2772,0.4088,0.0011,0.0003,7,False,4,True
linear_svr,K,"C=0.1, epsilon=0.25",0.5274,0.2786,0.4076,0.0011,0.0016,6,False,5,True
svr_linear,K,"C=0.01, epsilon=0.1",0.5278,0.2791,0.4082,0.0014,0.0020,4,False,5,False


## Selection

> the rule, applied to the table above: of the models tied with the best mean RMSE, the one with the lowest simplicity rank, and within that rank the lowest mean RMSE. The table is sorted that way, so the pick is the first row tied with the best. If no other model is within 0.002, the best model is the only one tied and it is the pick

In [38]:
# the best mean RMSE is always tied with itself, so there is at least one model to pick from
experimentM_best_model = experimentM_candidates["rmse"].idxmin()
experimentM_tied = experimentM_candidates[experimentM_candidates["tied with best"]]
assert experimentM_best_model in experimentM_tied.index

# sorted by simplicity rank, then mean RMSE: the first tied row is the simplest, and the lower RMSE within its rank
experimentM_chosen = experimentM_tied.index[0]
experimentM_chosen_row = experimentM_candidates.loc[experimentM_chosen]

print(f"best mean RMSE: {experimentM_best_rmse:.4f} ({experimentM_best_model}, {experimentM_candidates.loc[experimentM_best_model, 'experiment']})")
print(f"tied with the best (within {NOISE_FLOOR}), simplest first: "
      + ", ".join(f"{model_name} {rmse:.4f}" for model_name, rmse in experimentM_tied["rmse"].items()))
print(f"chosen: {experimentM_chosen} ({experimentM_chosen_row['experiment']}, simplicity rank {experimentM_chosen_row['simplicity rank']}), "
      f"setting: {experimentM_chosen_row['final setting']}, mean RMSE {experimentM_chosen_row['rmse']:.4f} "
      f"({experimentM_chosen_row['rmse'] - experimentM_best_rmse:.4f} above the best)")

best mean RMSE: 0.5258 (ridge, J)
tied with the best (within 0.002), simplest first: linear_regression 0.5258, ridge 0.5258, elastic_net 0.5258, lasso 0.5261, sgd 0.5261, linear_svr 0.5274
chosen: linear_regression (baseline, simplicity rank 1), setting: none (untuned), mean RMSE 0.5258 (0.0000 above the best)


## Final Pipeline

> the chosen model's pipeline (the fixed preprocessing + the model at its final setting), fit on all 1,374 complete rows. A tuned model reuses the final model its experiment already fit (`experimentJ_final_models`, `experimentK_final_models`, or `experimentL_final_models`), so nothing is refit; the untuned baseline has no final model in this notebook, so if it is chosen a clone of `baseline_pipeline` is fit on all the rows here. The pipeline is saved, reloaded from disk, and the reloaded one must predict the same values on the 1,374 rows as the one in memory (asserted), so the saved file is the model the test-set step will score

In [39]:
# the baseline is fit on all the rows here; a tuned model is the final model its experiment already fit and saved
experimentM_chosen_experiment = experimentM_chosen_row["experiment"]
if experimentM_chosen_experiment == "baseline":
	experimentM_final_model = clone(baseline_pipeline).fit(X_clean, y_clean)
else:
	_, _, _, experimentM_source_models = experimentM_sources[experimentM_chosen_experiment]
	experimentM_final_model = experimentM_source_models[experimentM_chosen]

experimentM_model_path = experimentM_model_output_dir / model_file_name(experimentM_chosen, preprocessing_name(preprocessor))
joblib.dump(experimentM_final_model, experimentM_model_path)

# the saved file must be the same model: reloaded from disk, it predicts what the in-memory pipeline predicts
experimentM_reloaded_model = joblib.load(experimentM_model_path)
assert np.allclose(experimentM_reloaded_model.predict(X_clean), experimentM_final_model.predict(X_clean)), "the reloaded pipeline predicts differently"

print(f"saved {experimentM_chosen} to {experimentM_model_path}")
print(f"reloaded: same predictions on all {len(X_clean):,} rows")
print(f"preprocessing: {preprocessing_name(experimentM_final_model.named_steps['preprocess'])}")
print(f"model: {experimentM_final_model.named_steps['model']!r}")
experimentM_final_model

saved linear_regression to /Users/mrt/Documents/MrT/IUBloomington/Fall 2026 - Applied Machine Learning/code/hw/Applied-ML-HW1/Q1/MODELS/M-FINAL-DropRowsWithMissingValues/linear_regression__transform-ColumnTransformer(PowerTransformer(method=yeo-johnson,standardize=False)[Social,Life exp,Freedom,Generosity,Corruption,Neg affect])__scale-StandardScaler.joblib
reloaded: same predictions on all 1,374 rows
preprocessing: transform-ColumnTransformer(PowerTransformer(method=yeo-johnson,standardize=False)[Social,Life exp,Freedom,Generosity,Corruption,Neg affect])__scale-StandardScaler
model: LinearRegression()


Pipeline(steps=[('preprocess',
                 Pipeline(steps=[('impute', 'passthrough'),
                                 ('transform',
                                  ColumnTransformer(remainder='passthrough',
                                                    transformers=[('transform',
                                                                   PowerTransformer(standardize=False),
                                                                   ['Social '
                                                                    'support',
                                                                    'Healthy '
                                                                    'life '
                                                                    'expectancy '
                                                                    'at birth',
                                                                    'Freedom '
                                                                    'to make '
                                                                    'life '
                                                                    'choices',
                                                                    'Generosity',
                                                                    'Perceptions '
                                                                    'of '
                                                                    'corruption',
                                                                    'Negative '
                                                                    'affect'])])),
                                 ('scale', StandardScaler())])),
                ('model', LinearRegression())])

## Experiment M Results

- **The rule picks untuned linear_regression:** RMSE 0.5258, MSE 0.2769, MAE 0.4084 over the 20 folds (seed std 0.0009), 0.0000 above the best mean RMSE (ridge from J, also 0.5258). The selection is made by code from the rule set before any results, not by hand.
- **Six of the seven candidates tie with the best:** linear_regression, ridge, ElasticNet (0.5258), Lasso, SGD (0.5261), and LinearSVR (0.5274) are all within 0.002 of ridge, so the simplicity order decides and linear_regression ranks first. svr_linear (0.5278) falls just outside the cutoff because it is measured from ridge's slightly lower score.
- **No tuned model beats the bar:** the best gaps vs the baseline are ridge and ElasticNet at -0.0000 (12 of 20 folds each, a coin flip). The candidates table matches the J, K, and L summaries exactly, so M is on the same scale as every earlier experiment.
- **LinearSVR has the lowest MAE (0.4076, 0.0008 better than the pick)**, but the rule uses RMSE and LinearSVR is 0.0016 worse on it, so the trade does not change the decision.
- **The final pipeline is saved and checked:** drop incomplete rows → Yeo-Johnson (standardize=False) on the 6 skewed columns → StandardScaler → LinearRegression, fit on all 1,374 complete rows and saved to `MODELS/M-FINAL-DropRowsWithMissingValues/`. Reloading it from disk gives the same predictions on all 1,374 rows.
- **Carry forward**
  - **Where the gain came from:** on the same seed-42 folds, A's untouched linear_regression scored 0.547 and F's Yeo-Johnson + StandardScaler scored 0.527. That transform accounts for the whole 0.020 improvement; scaling (B, F), imputation (G), per-column transform choices (E), and tuning (J to L) added nothing beyond the 0.002 noise floor.
  - **For the test-set step:** load the saved pipeline and score it once on the held-back test set (334 rows, none with missing values, so no rows are dropped) with the same `SCORERS`. Expect test RMSE near 0.526; a single 334-row test set will vary more than the 20-fold mean, so a difference of a few hundredths is not a reason to go back and retune.